# Stage

- Internal Stage
- External Stage

- internal stage: These are of 3 types : 
1. User Stage
2. Table Stage
3. Named Internal Stage

User Stage : (@~)
- it is allocated to each user to store files
- to store files that are staged and managed by a single user but can be loaded into multiple tables.
- cannot be altered or dropped.
- this is not appropriate if multiple users want to access the files.

TABLE STAGE (@%)
- a table stage is available for each table created in snowflake
- table stage have same name as table
- to store files that are staged and managed by one or more users but only loaded into a single table
- table stages cannot be altered or dropped.
- table stage is not a seperate database object rather it is an implicit stage tied to the table itself.

NAMED INTERNAL STAGE (@)
- A named internal stage is a database object created in a schema
- to store files that are staged and managed by one or more users and loaded into one or more tables.
- create stages using the CREATE STAGE command.

External Stage
An external stage is the external cloud storage location where data files are stored.
- Loading data from any of the following cloud storage services is supported regardless of the cloud platform that hosts your snowflake account: s3, GCS, azure
- an external stage is a database object created in a schema
- this object stores url to files in cloud storage

Server side encryption
by default, all data at rest is automatically encrypted using AES-256 bit encryption, and all data in transit is protected via TLS 1.2

1. Hierarchial key model
- root key: master of masters stored securely in Hardware security module provieded by cloud host. It never leaves HSM.
- account master key : unique to your specific snowflake account
- table master key : unique to each individual table
- file key : every single micro partition file has its own unique encryption key.
2. Automatic maintainance
- Key rotation
- periodic rekeying
3. Advanced control (tri secret secure)
- snowflake managed key
- customer managed key
- user creds
4. Encryption for staging data: you can use snowflake's sse or your own cloud provider's encryption

# Directory table
allows you to query a stage as if it were a database table. A directory table is not a seperate object. it is a feature you enbale on snowflake stage. Once enabled, it stores a snpashot of the files in that stage, including
- rleative path: file location
- file size: how big the file is
- last modified: when it was uploaded
- etag: file's unique versioin identifier
- file url: a scoped url used to download or view the file

Why use them?
These are the backbone of Unstructured Data Management in Snowflake.

use case
- Data Pipelines: using a select statement on the directory table to find only files uploaded in the last hour to trigger a processing job
- AI/ML: oassing file URLs to snowflake cortex AI to perform OCR on PDFs or image recogn.
- Governance: quickly auditing how muych storage your unstructured fikes are taking across diff departments.

syntax
create stage my_ext_stage
url ='s3://my-bucket/'
storage_integration = my_s3_int
directory = (enable=true);

select * from directory(@my_external_stage)

# COPY into command

VALIDATION MODE
copy into <table_name>
from @externalstage
files = ('<file_name>','<file_name2>')
file_format=<file_format_name>
validation_mode = return_n_rows | return_errors |return_all_errors ;

- validate the data files instead of loading them into the table
- return_errors give all errors in the files
- reurn_all_errors gives all errors from previously loaded files if we have used on_error = continue
- return n rows, displays first n records and fails at the first error record

RETURN FAILED ONLY

copy into <table_name>
from @externalstage
files = ('<file_name>','<file_name2>')
file_format=<file_format_name>
return_failed_only = true | false ;

ON ERROR

copy into <table_name>
from @externalstage
files = ('<file_name>','<file_name2>')
file_format=<file_format_name>
on_error = continue | skip_file |skip_file_num |skip_file_num% |abort_statement  ;

continue - to skip error records and load remaining records
skip_file - to skip file that constains errors
skip_file_num - skip a file when the no of erros is >=specified numebr
skip_file_num% - skip a file when the percentage of errors exceeds the specified percentage
default is abort statement, abort the load operation if any error is found in data file

FORCE & SIZE_lIMIT

copy into <table_name>
from @externalstage
files = ('<file_name>','<file_name2>')
file_format=<file_format_name>
force = true | false
size limit = <number> ;

- load all files regardless of whether they have been loaded
- default is false, if we don't specify this property copy command will not fail but it skips loading the data


- specify maximum size in bytes of data loaded in that command.
- when the threshold is exceeded, the copy command stops loading
- in case of multiple files of same pattern, it wwill stop after the size limit, some files will be laoded fully and other partially

TRUNCATE COLUMNS or ENFORCE_LENGTH

copy into <table_name>
from @externalstage
files = ('<file_name>','<file_name2>')
file_format=<file_format_name>
truncatecolumns = true | false
enforce_length = true | false;

- specifies whether to truncate text strings that exceeds the target column length
- default is false, that means if we don't speicify this option and if text strings that exceeds the target column length, then copy command will fail

PURGE

copy into <table_name>
from @externalstage
files = ('<file_name>','<file_name2>')
file_format=<file_format_name>
purge = true | false

- specifies whether to remove the data files from the stage automatically after the data has been loaded succesfulyy.
- default is false

LOAD UNCERTAIN FILES

copy into <table_name>
from @externalstage
files = ('<file_name>','<file_name2>')
file_format=<file_format_name>
load_uncertain_files = true | false

- specifies to laod files for which the load status is unknown. copy command skips these files by default

load status is unknown if
- last mod date is older than 64 days
- initial set of data was loaded more than 64 days earlier
- if file got loaded into table, the event occured 64 days earlier.

ERROR HANDLING

1. error handling in data loading: when bulk loading, it is better to use on_error, it determines how the system reacts when encountered a malfunction row.
2. hadnled erros in SQL scripting:
if you are writing snowflake scripting or Store Procedure, you use exception handling simialr to python or java

